# 05 · On-policy distillation: OPD → OPSD → MOPD

Keep RL's on-policy sampling, but replace the 0/1 reward with a **per-token grade
from a teacher**:

    advantage[t] = log p_teacher(token_t) − log p_student(token_t)      (= −reverse KL)

The student samples, the teacher says, at every token, "how likely would *I* have
been to write that?" Same `importance_sampling` loss as RL, but the signal is dense.

Three variants. **The loop is identical; only the teacher changes:**

| | teacher | teacher sees |
|---|---|---|
| **OPD**  | bigger model | the same prompt + "be concise" |
| **OPSD** | *the student itself* (frozen copy) | the prompt **+ the reference solution** |
| **MOPD** | a different expert per domain | the same prompt (+ "be concise" for math) |

In [ ]:
from common import *

GROUP_SIZE, BATCH_SIZE, STEPS, LR = 4, 16, 15, 1e-4
sampling_params = types.SamplingParams(max_tokens=MAX_TOKENS, temperature=1.0, stop=renderer.get_stop_sequences())
START_FROM = load_checkpoint("sft")
gsm8k_train = load_gsm8k("train").shuffle(seed=2)

In [ ]:
# ── LIVE: the generic on-policy distillation loop ─────────────────────────
def on_policy_distill(get_batch, label, steps=STEPS):
    """get_batch(step) -> list of (student_prompt, teacher_client, teacher_prompt)."""
    training_client = new_training_client(START_FROM)
    for step in range(steps):
        student = training_client.save_weights_and_get_sampling_client().result()
        items = get_batch(step)
        sample_futs = [student.sample(sp, GROUP_SIZE, sampling_params) for sp, _, _ in items]

        # Teacher scores the student's tokens. It doesn't generate anything.
        scored = []
        for (student_prompt, teacher, teacher_prompt), fut in zip(items, sample_futs):
            for s in fut.result().sequences:
                full = teacher_prompt.append(types.EncodedTextChunk(tokens=s.tokens))
                tfut = teacher.compute_logprobs(full, prompt_logprobs_last_n=len(s.tokens))
                scored.append((student_prompt, s, tfut))

        batch, kls = [], []
        for student_prompt, s, tfut in scored:
            teacher_lp = tfut.result()[-len(s.tokens):]
            advantages = [t - st for t, st in zip(teacher_lp, s.logprobs)]  # per-token!
            kls.append(-sum(advantages) / len(advantages))
            batch.append(make_rl_datum(student_prompt, s.tokens, s.logprobs, advantages))

        training_client.forward_backward(batch, loss_fn="importance_sampling")
        training_client.optim_step(adam(LR)).result()
        print(f"[{label}] step {step:2d}  reverse KL to teacher = {sum(kls) / len(kls):.3f}")
    return training_client


def gsm8k_rows(step):
    return gsm8k_train.select(range(step * BATCH_SIZE, (step + 1) * BATCH_SIZE))


def student_prompt(question):
    return renderer.build_generation_prompt(gsm8k_convo(question))


def concise_teacher_prompt(question):
    # The teacher is *told* to be brief, so it gives high logprobs to short, direct solutions.
    # Without this, a chatty teacher would pull the student past our 256-token budget.
    convo = gsm8k_convo(question)
    convo[-1] = {**convo[-1], "content": convo[-1]["content"] + " Be concise: solve it in a few short sentences."}
    return renderer.build_generation_prompt(convo)

## OPD: on-policy distillation from a bigger teacher

The student sees the normal prompt; the teacher sees it plus "be concise".

In [ ]:
# ── LIVE ──────────────────────────────────────────────────────────────────
math_teacher = service_client.create_sampling_client(base_model=MATH_TEACHER).result()

def opd_batch(step):
    return [(student_prompt(q), math_teacher, concise_teacher_prompt(q)) for q in gsm8k_rows(step)["question"]]

tc = on_policy_distill(opd_batch, "OPD")
eval_gsm8k(tc.save_weights_and_get_sampling_client().result(), label="5a. OPD (35B-A3B teacher)")

## OPSD: on-policy *self*-distillation

No bigger model. The teacher is a frozen copy of the student, but it gets to
**peek at the reference solution** (privileged information). The student learns
to produce, without the answer key, what it would have written *with* it.

In [ ]:
# ── LIVE: only the teacher + teacher prompt change ────────────────────────
self_teacher = new_training_client(START_FROM).save_weights_and_get_sampling_client().result()

def teacher_prompt_with_solution(question, answer):
    hint = f"{question}\n\n(Reference solution, for you only: {gsm8k_solution(answer)})"
    return renderer.build_generation_prompt([*FEWSHOT, {"role": "user", "content": hint + QUESTION_SUFFIX}])

def opsd_batch(step):
    rows = gsm8k_rows(step)
    return [
        (student_prompt(q), self_teacher, teacher_prompt_with_solution(q, a))
        for q, a in zip(rows["question"], rows["answer"])
    ]

tc = on_policy_distill(opsd_batch, "OPSD")
eval_gsm8k(tc.save_weights_and_get_sampling_client().result(), label="5b. OPSD (self + answer key)")

## MOPD: multi-teacher on-policy distillation

One student, several domain experts. Each prompt is routed to its domain's
teacher. Here: math → Qwen3.6-35B-A3B (told to be concise), general chat → Qwen3.5-397B-A17B.
Expect the math score to dip: the chat teacher has no 256-token budget, so half the
training signal pulls the student toward longer answers. That's the trade-off of mixing teachers.

In [ ]:
# ── LIVE: a router picks the teacher ──────────────────────────────────────
chat_teacher = service_client.create_sampling_client(base_model=CHAT_TEACHER).result()
chat = datasets.load_dataset("HuggingFaceH4/no_robots", split="train").shuffle(seed=0)
TEACHERS = {"math": math_teacher, "chat": chat_teacher}

def mopd_batch(step):
    half = BATCH_SIZE // 2
    items = [("math", q) for q in gsm8k_rows(step)["question"][:half]]
    for msgs in chat.select(range(step * half, (step + 1) * half))["messages"]:
        user_turn = next(m for m in msgs if m["role"] == "user")
        items.append(("chat", user_turn))
    out = []
    for domain, x in items:
        if domain == "math":
            out.append((student_prompt(x), TEACHERS["math"], concise_teacher_prompt(x)))
        else:
            p = renderer.build_generation_prompt([x])
            out.append((p, TEACHERS["chat"], p))
    return out

tc = on_policy_distill(mopd_batch, "MOPD")
mopd_client = tc.save_weights_and_get_sampling_client().result()
eval_gsm8k(mopd_client, label="5c. MOPD (math + chat)")
show("MOPD chat sample", generate(mopd_client, "Write a two-line poem about GPUs."))

## Scoreboard

In [ ]:
scoreboard()

**Takeaways**
- Distillation = RL where the reward is *dense* (every token) and comes from a model, not a verifier.
- OPD needs a stronger teacher. OPSD needs only privileged info (answer key, docs, tools).
  MOPD lets one student absorb several specialists at once.
- In Tinker, all three were the *same 25-line loop*, swapping only `(teacher, teacher_prompt)`.